In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os
from datasets import load_dataset, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel, RobertaModel
from dataclasses import dataclass
from typing import Optional, Tuple, Any, Dict, List
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda' 

In [ ]:
MODEL_NAME = "roberta-large"
MAX_LENGTH = 512
N_FOLDS = 7

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

clarity_label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
clarity_id2label = {v: k for k, v in clarity_label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

evasion_classes = [
    'Declining to answer', 'Dodging', 'General', 'Explicit', 
    'Claims ignorance', 'Clarification', 'Implicit', 
    'Partial/half-answer', 'Deflection'
]
evasion_label2id = {label: idx for idx, label in enumerate(evasion_classes)}
evasion_id2label = {v: k for k, v in evasion_label2id.items()}

In [ ]:
ds = load_dataset("ailsntua/QEvasion")
train_full_df = ds['train'].to_pandas()
dev_df = ds['test'].to_pandas()

print(f"Train columns: {train_full_df.columns.tolist()}")
print(f"Train size: {len(train_full_df)}")
print(f"\nClarity label distribution (train):")
print(train_full_df['clarity_label'].value_counts())
print(f"\nEvasion label distribution (train):")
print(train_full_df['evasion_label'].value_counts())

print(f"\nDev columns: {dev_df.columns.tolist()}")
print(f"Dev size: {len(dev_df)}")
if "clarity_label" in dev_df.columns:
    print(f"\nClarity label distribution (dev):")
    print(dev_df['clarity_label'].value_counts())
else:
    print("\nNo clarity_label column found in dev split.")

annotator_cols = [c for c in ["annotator1", "annotator2", "annotator3"] if c in dev_df.columns]
print(f"\nDev annotator columns for evasion: {annotator_cols if annotator_cols else 'None'}")

In [ ]:
def tokenize_function_train(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
]
    tokenized = tokenizer(
        inputs,
        truncation=False,  
        padding=False,
        max_length=None,
    )
    
    tokenized["clarity_labels"] = [clarity_label2id[l] for l in examples["clarity_label"]]
    tokenized["evasion_labels"] = [evasion_label2id[l] for l in examples["evasion_label"]]
    
    return tokenized


def tokenize_function_dev(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
]
    tokenized = tokenizer(
        inputs,
        truncation=False, 
        padding=False,
        max_length=None,
    )
    
    if "clarity_label" in examples:
        tokenized["clarity_labels"] = [clarity_label2id[l] for l in examples["clarity_label"]]
    else:
        tokenized["clarity_labels"] = [-100] * len(examples["question"])
    
    # Dev split has no single gold evasion label; mask this head during prediction/eval.
    tokenized["evasion_labels"] = [-100] * len(examples["question"])
    
    return tokenized

train_full_dataset = Dataset.from_pandas(train_full_df, preserve_index=False)
train_tokenized_full = train_full_dataset.map(
    tokenize_function_train, 
    batched=True, 
    remove_columns=train_full_dataset.column_names
)

dev_dataset = Dataset.from_pandas(dev_df, preserve_index=False)
dev_tokenized = dev_dataset.map(
    tokenize_function_dev, 
    batched=True, 
    remove_columns=dev_dataset.column_names
)

print(f"Train tokenized: {len(train_tokenized_full)} samples")
print(f"Dev tokenized: {len(dev_tokenized)} samples")

In [ ]:
@dataclass
class HierarchicalMultiHeadDataCollator:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        clarity_labels = torch.tensor(
            [f.pop("clarity_labels") for f in features], 
            dtype=torch.long
        )
        evasion_labels = torch.tensor(
            [f.pop("evasion_labels") for f in features], 
            dtype=torch.long
        )
        
        batch = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt"
        )
        batch["clarity_labels"] = clarity_labels
        batch["evasion_labels"] = evasion_labels
        
        return batch

data_collator = HierarchicalMultiHeadDataCollator(tokenizer=tokenizer)

In [ ]:
@dataclass
class MultiHeadClassifierOutput:
    loss: Optional[torch.FloatTensor] = None
    logits_clarity: torch.FloatTensor = None
    logits_evasion: torch.FloatTensor = None
    hidden_states: Optional[Tuple[torch.FloatTensor, ...]] = None
    attentions: Optional[Tuple[torch.FloatTensor, ...]] = None


class MultiHeadHierarchicalMaxPooling(RobertaPreTrainedModel):
    
    @classmethod
    def _can_set_experts_implementation(cls) -> bool:
        return False
    
    def __init__(self, config):
        super().__init__(config)
        self.num_clarity_labels = 3
        self.num_evasion_labels = 9
        self.config = config

        self.roberta = RobertaModel(config)

        classifier_dropout = (
            config.classifier_dropout 
            if hasattr(config, 'classifier_dropout') and config.classifier_dropout is not None 
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)

        self.classifier_clarity = nn.Linear(config.hidden_size, self.num_clarity_labels)
        self.classifier_evasion = nn.Linear(config.hidden_size, self.num_evasion_labels)

        self.post_init()
    
    def create_chunks_batched(self, input_ids, attention_mask, max_length=512, stride=256):
        batch_size, seq_len = input_ids.shape
        
        all_chunk_ids = []
        all_chunk_masks = []
        chunk_to_sample = [] 
        
        for sample_idx in range(batch_size):
            sample_input_ids = input_ids[sample_idx]
            sample_attention_mask = attention_mask[sample_idx]

            actual_length = sample_attention_mask.sum().item()

            if actual_length <= max_length:
                chunk_ids = sample_input_ids[:max_length]
                chunk_mask = sample_attention_mask[:max_length]
                
                if len(chunk_ids) < max_length:
                    padding_length = max_length - len(chunk_ids)
                    chunk_ids = torch.cat([
                        chunk_ids,
                        torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                    ])
                    chunk_mask = torch.cat([
                        chunk_mask,
                        torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)
                    ])
                
                all_chunk_ids.append(chunk_ids)
                all_chunk_masks.append(chunk_mask)
                chunk_to_sample.append(sample_idx)
            else:
                start = 0
                while start < actual_length:
                    end = min(start + max_length, actual_length)
                    
                    chunk_ids = sample_input_ids[start:end]
                    chunk_mask = sample_attention_mask[start:end]
                    
                    if len(chunk_ids) < max_length:
                        padding_length = max_length - len(chunk_ids)
                        chunk_ids = torch.cat([
                            chunk_ids,
                            torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                        ])
                        chunk_mask = torch.cat([
                            chunk_mask,
                            torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)
                        ])
                    
                    all_chunk_ids.append(chunk_ids)
                    all_chunk_masks.append(chunk_mask)
                    chunk_to_sample.append(sample_idx)
  
                    start += stride
                    if end >= actual_length:
                        break

        all_chunk_ids = torch.stack(all_chunk_ids, dim=0)
        all_chunk_masks = torch.stack(all_chunk_masks, dim=0)
        chunk_to_sample = torch.tensor(chunk_to_sample, dtype=torch.long, device=input_ids.device)
        
        return all_chunk_ids, all_chunk_masks, chunk_to_sample
    
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        clarity_labels=None,
        evasion_labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ) -> MultiHeadClassifierOutput:
        
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        
        batch_size = input_ids.shape[0]

        all_chunk_ids, all_chunk_masks, chunk_to_sample = self.create_chunks_batched(
            input_ids, attention_mask, max_length=512, stride=256
        )

        outputs = self.roberta(
            input_ids=all_chunk_ids,
            attention_mask=all_chunk_masks,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
        )

        all_cls_embeddings = outputs.last_hidden_state[:, 0, :]
        batch_embeddings = []
        for sample_idx in range(batch_size):
            sample_mask = chunk_to_sample == sample_idx
            sample_chunk_embeddings = all_cls_embeddings[sample_mask]  

            pooled_embedding = torch.max(sample_chunk_embeddings, dim=0)[0]  
            batch_embeddings.append(pooled_embedding)

        pooled_output = torch.stack(batch_embeddings, dim=0)
        pooled_output = self.dropout(pooled_output)

        logits_clarity = self.classifier_clarity(pooled_output)
        logits_evasion = self.classifier_evasion(pooled_output)

        loss = None
        if clarity_labels is not None and evasion_labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss_clarity = loss_fct(logits_clarity.view(-1, self.num_clarity_labels), clarity_labels.view(-1))
            loss_evasion = loss_fct(logits_evasion.view(-1, self.num_evasion_labels), evasion_labels.view(-1))
            loss = loss_clarity + loss_evasion
        
        return MultiHeadClassifierOutput(
            loss=loss,
            logits_clarity=logits_clarity,
            logits_evasion=logits_evasion,
            hidden_states=None,
            attentions=None,
        )

In [ ]:
def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(predictions, tuple):
        logits_clarity, logits_evasion = predictions
        if hasattr(logits_clarity, 'cpu'):
            logits_clarity = logits_clarity.cpu().numpy()
        if hasattr(logits_evasion, 'cpu'):
            logits_evasion = logits_evasion.cpu().numpy()
    else:
        logits_clarity = predictions[:, :3]
        logits_evasion = predictions[:, 3:]
    preds_clarity = np.argmax(logits_clarity, axis=-1)
    preds_evasion = np.argmax(logits_evasion, axis=-1)
    if isinstance(labels, tuple):
        labels_clarity, labels_evasion = labels
        if hasattr(labels_clarity, 'cpu'):
            labels_clarity = labels_clarity.cpu().numpy()
        if hasattr(labels_evasion, 'cpu'):
            labels_evasion = labels_evasion.cpu().numpy()
    else:
        labels_clarity = labels[:, 0] if labels.ndim > 1 else labels
        labels_evasion = labels[:, 1] if labels.ndim > 1 else labels

    valid_evasion_mask = labels_evasion != -100

    accuracy_clarity = accuracy_score(labels_clarity, preds_clarity)
    f1_clarity = f1_score(labels_clarity, preds_clarity, average='macro')

    if valid_evasion_mask.sum() > 0:
        accuracy_evasion = accuracy_score(labels_evasion[valid_evasion_mask], preds_evasion[valid_evasion_mask])
        f1_evasion = f1_score(labels_evasion[valid_evasion_mask], preds_evasion[valid_evasion_mask], average='macro')
    else:
        accuracy_evasion = 0.0
        f1_evasion = 0.0
    
    return {
        'accuracy_clarity': accuracy_clarity,
        'f1_clarity': f1_clarity,
        'accuracy_evasion': accuracy_evasion,
        'f1_evasion': f1_evasion,
        'f1_combined': (f1_clarity + f1_evasion) / 2,
    }

In [ ]:
class MultiHeadTrainer(Trainer):
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss
    
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        clarity_labels = inputs.get('clarity_labels')
        evasion_labels = inputs.get('evasion_labels')
        
        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs.loss
            logits_clarity = outputs.logits_clarity
            logits_evasion = outputs.logits_evasion
        
        if prediction_loss_only:
            return (loss, None, None)
        
        logits = (logits_clarity.detach().float(), logits_evasion.detach().float())

        if clarity_labels is not None and evasion_labels is not None:
            labels = (clarity_labels.detach(), evasion_labels.detach())
        else:
            labels = None
        
        return (loss, logits, labels)


class EarlyStoppingCallback(TrainerCallback):    
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1_combined", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_dev_probs_clarity = []
fold_dev_probs_evasion = []

fold_oof_probs_clarity = []
fold_oof_probs_evasion = []
fold_oof_indices = []

fold_metrics = []

best_fold = None
best_f1_combined = 0
best_model_state = None

for fold, (train_idx, val_idx) in enumerate(skf.split(train_full_df, train_full_df['clarity_label']), 1):
    print(f"\nFold {fold}/{N_FOLDS}")
    print(f"{'='*30}")
    
    train_dataset = train_tokenized_full.select(train_idx)
    val_dataset = train_tokenized_full.select(val_idx)
    
    print(f"Train samples: {len(train_dataset)}")
    print(f"Val samples: {len(val_dataset)}")
    
    config = AutoConfig.from_pretrained(MODEL_NAME)
    
    model = MultiHeadHierarchicalMaxPooling(config)
    
    base_model = AutoModel.from_pretrained(MODEL_NAME)
    model.roberta.load_state_dict(base_model.state_dict())
    del base_model
    
    training_args = TrainingArguments(
        output_dir=f"./results_multihead_maxpool_dev_fold{fold}",
        learning_rate=1e-5,
        per_device_train_batch_size=8, 
        per_device_eval_batch_size=8,
        num_train_epochs=15,
        warmup_ratio=0.1,
        weight_decay=0.01,
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        bf16=True,
        bf16_full_eval=True,
        optim="adamw_torch",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_combined",
        greater_is_better=True,
        logging_steps=50,
        seed=SEED + fold,
        disable_tqdm=True,
        report_to="none",
    )
    
    trainer = MultiHeadTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1_combined")],
    )

    trainer.train()

    val_results = trainer.evaluate()
    fold_metrics.append({
        'fold': fold,
        'val_accuracy_clarity': val_results['eval_accuracy_clarity'],
        'val_f1_clarity': val_results['eval_f1_clarity'],
        'val_accuracy_evasion': val_results['eval_accuracy_evasion'],
        'val_f1_evasion': val_results['eval_f1_evasion'],
        'val_f1_combined': val_results['eval_f1_combined'],
    })
    
    # Track best model
    if val_results['eval_f1_combined'] > best_f1_combined:
        best_f1_combined = val_results['eval_f1_combined']
        best_fold = fold
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    print(f"Fold {fold} Results:")
    print(f"  Clarity - Accuracy: {val_results['eval_accuracy_clarity']:.4f}, F1: {val_results['eval_f1_clarity']:.4f}")
    print(f"  Evasion - Accuracy: {val_results['eval_accuracy_evasion']:.4f}, F1: {val_results['eval_f1_evasion']:.4f}")
    print(f"  Combined F1: {val_results['eval_f1_combined']:.4f}")
    
    oof_output = trainer.predict(val_dataset)
    logits_clarity_oof, logits_evasion_oof = oof_output.predictions

    oof_probs_clarity = torch.nn.functional.softmax(torch.tensor(logits_clarity_oof), dim=-1).numpy()
    oof_probs_evasion = torch.nn.functional.softmax(torch.tensor(logits_evasion_oof), dim=-1).numpy()
    
    fold_oof_probs_clarity.append(oof_probs_clarity)
    fold_oof_probs_evasion.append(oof_probs_evasion)
    fold_oof_indices.append(val_idx)

    dev_output = trainer.predict(dev_tokenized)
    logits_clarity_dev, logits_evasion_dev = dev_output.predictions
    
    dev_probs_clarity = torch.nn.functional.softmax(torch.tensor(logits_clarity_dev), dim=-1).numpy()
    dev_probs_evasion = torch.nn.functional.softmax(torch.tensor(logits_evasion_dev), dim=-1).numpy()
    
    fold_dev_probs_clarity.append(dev_probs_clarity)
    fold_dev_probs_evasion.append(dev_probs_evasion)
    
    del model, trainer
    torch.cuda.empty_cache()

print(f"\nBest fold: {best_fold} with F1 Combined: {best_f1_combined:.4f}")

In [ ]:
print("Cross-Validation Metrics")

metrics_df = pd.DataFrame(fold_metrics)
print(metrics_df.to_string(index=False))

print(f"\nClarity Head:")
print(f"Average Validation Accuracy: {metrics_df['val_accuracy_clarity'].mean():.4f} ± {metrics_df['val_accuracy_clarity'].std():.4f}")
print(f"Average Validation Macro F1: {metrics_df['val_f1_clarity'].mean():.4f} ± {metrics_df['val_f1_clarity'].std():.4f}")

print(f"\nEvasion Head:")
print(f"Average Validation Accuracy: {metrics_df['val_accuracy_evasion'].mean():.4f} ± {metrics_df['val_accuracy_evasion'].std():.4f}")
print(f"Average Validation Macro F1: {metrics_df['val_f1_evasion'].mean():.4f} ± {metrics_df['val_f1_evasion'].std():.4f}")

print(f"\nCombined:")
print(f"Average Combined F1: {metrics_df['val_f1_combined'].mean():.4f} ± {metrics_df['val_f1_combined'].std():.4f}")

In [ ]:
print("Full training set OOF analysis")

oof_probs_clarity_full = np.zeros((len(train_full_df), 3))
oof_probs_evasion_full = np.zeros((len(train_full_df), 9))

for probs_c, probs_e, idxs in zip(fold_oof_probs_clarity, fold_oof_probs_evasion, fold_oof_indices):
    oof_probs_clarity_full[idxs] = probs_c
    oof_probs_evasion_full[idxs] = probs_e

oof_preds_clarity = np.argmax(oof_probs_clarity_full, axis=1)
oof_preds_evasion = np.argmax(oof_probs_evasion_full, axis=1)

oof_pred_labels_clarity = [clarity_id2label[p] for p in oof_preds_clarity]
oof_pred_labels_evasion = [evasion_id2label[p] for p in oof_preds_evasion]

y_true_train_clarity = train_full_df['clarity_label'].tolist()
y_true_train_evasion = train_full_df['evasion_label'].tolist()

print("Clarity head (OOF)")
print(classification_report(y_true_train_clarity, oof_pred_labels_clarity, labels=clarity_labels, zero_division=0))

print("\nEvasion head (OOF)")
print(classification_report(y_true_train_evasion, oof_pred_labels_evasion, labels=evasion_classes, zero_division=0))

In [ ]:
import textwrap
import json
import re
from pathlib import Path


def safe_col_suffix(label: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")


print("Building full OOF analysis artifacts")

oof_probs_clarity_full = np.zeros((len(train_full_df), 3))
oof_probs_evasion_full = np.zeros((len(train_full_df), 9))
oof_fold_id = np.full(len(train_full_df), -1, dtype=int)

for fold_id, (probs_c, probs_e, idxs) in enumerate(
    zip(fold_oof_probs_clarity, fold_oof_probs_evasion, fold_oof_indices), start=1
):
    oof_probs_clarity_full[idxs] = probs_c
    oof_probs_evasion_full[idxs] = probs_e
    oof_fold_id[idxs] = fold_id

oof_analysis_df = train_full_df.copy().reset_index(drop=True)
oof_analysis_df["row_id"] = np.arange(len(oof_analysis_df))
oof_analysis_df["oof_fold"] = oof_fold_id

oof_analysis_df["gold_clarity"] = oof_analysis_df["clarity_label"]
oof_analysis_df["gold_evasion"] = oof_analysis_df["evasion_label"]
oof_analysis_df["gold_clarity_id"] = oof_analysis_df["gold_clarity"].map(clarity_label2id)
oof_analysis_df["gold_evasion_id"] = oof_analysis_df["gold_evasion"].map(evasion_label2id)

if oof_analysis_df["gold_clarity_id"].isna().any() or oof_analysis_df["gold_evasion_id"].isna().any():
    raise ValueError("Found unmapped gold labels in OOF table.")

oof_analysis_df["gold_clarity_id"] = oof_analysis_df["gold_clarity_id"].astype(int)
oof_analysis_df["gold_evasion_id"] = oof_analysis_df["gold_evasion_id"].astype(int)

oof_analysis_df["pred_clarity_id"] = np.argmax(oof_probs_clarity_full, axis=1)
oof_analysis_df["pred_evasion_id"] = np.argmax(oof_probs_evasion_full, axis=1)
oof_analysis_df["pred_clarity"] = [clarity_id2label[int(i)] for i in oof_analysis_df["pred_clarity_id"]]
oof_analysis_df["pred_evasion"] = [evasion_id2label[int(i)] for i in oof_analysis_df["pred_evasion_id"]]

oof_analysis_df["pred_clarity_conf"] = oof_probs_clarity_full.max(axis=1)
oof_analysis_df["pred_evasion_conf"] = oof_probs_evasion_full.max(axis=1)

# Save full per-class probabilities for complete downstream analysis.
for class_id, class_label in clarity_id2label.items():
    suffix = f"{class_id}_{safe_col_suffix(class_label)}"
    oof_analysis_df[f"prob_clarity_{suffix}"] = oof_probs_clarity_full[:, int(class_id)]

for class_id, class_label in evasion_id2label.items():
    suffix = f"{class_id}_{safe_col_suffix(class_label)}"
    oof_analysis_df[f"prob_evasion_{suffix}"] = oof_probs_evasion_full[:, int(class_id)]

clarity_sorted = np.sort(oof_probs_clarity_full, axis=1)
evasion_sorted = np.sort(oof_probs_evasion_full, axis=1)
oof_analysis_df["pred_clarity_conf_2nd"] = clarity_sorted[:, -2]
oof_analysis_df["pred_evasion_conf_2nd"] = evasion_sorted[:, -2]
oof_analysis_df["pred_clarity_margin"] = oof_analysis_df["pred_clarity_conf"] - oof_analysis_df["pred_clarity_conf_2nd"]
oof_analysis_df["pred_evasion_margin"] = oof_analysis_df["pred_evasion_conf"] - oof_analysis_df["pred_evasion_conf_2nd"]

eps = 1e-12
oof_analysis_df["pred_clarity_entropy"] = -(
    oof_probs_clarity_full * np.log(np.clip(oof_probs_clarity_full, eps, 1.0))
).sum(axis=1)
oof_analysis_df["pred_evasion_entropy"] = -(
    oof_probs_evasion_full * np.log(np.clip(oof_probs_evasion_full, eps, 1.0))
).sum(axis=1)
oof_analysis_df["pred_clarity_entropy_norm"] = oof_analysis_df["pred_clarity_entropy"] / np.log(oof_probs_clarity_full.shape[1])
oof_analysis_df["pred_evasion_entropy_norm"] = oof_analysis_df["pred_evasion_entropy"] / np.log(oof_probs_evasion_full.shape[1])

clarity_top3_idx = np.argsort(oof_probs_clarity_full, axis=1)[:, -3:][:, ::-1]
evasion_top3_idx = np.argsort(oof_probs_evasion_full, axis=1)[:, -3:][:, ::-1]

oof_analysis_df["top3_clarity_labels"] = [
    json.dumps([clarity_id2label[int(i)] for i in idx_row], ensure_ascii=True)
    for idx_row in clarity_top3_idx
]
oof_analysis_df["top3_evasion_labels"] = [
    json.dumps([evasion_id2label[int(i)] for i in idx_row], ensure_ascii=True)
    for idx_row in evasion_top3_idx
]
oof_analysis_df["top3_clarity_probs"] = [
    json.dumps([float(np.round(oof_probs_clarity_full[row_idx, int(i)], 6)) for i in idx_row])
    for row_idx, idx_row in enumerate(clarity_top3_idx)
]
oof_analysis_df["top3_evasion_probs"] = [
    json.dumps([float(np.round(oof_probs_evasion_full[row_idx, int(i)], 6)) for i in idx_row])
    for row_idx, idx_row in enumerate(evasion_top3_idx)
]

row_positions = np.arange(len(oof_analysis_df))
gold_clarity_probs = oof_probs_clarity_full[row_positions, oof_analysis_df["gold_clarity_id"].to_numpy()]
gold_evasion_probs = oof_probs_evasion_full[row_positions, oof_analysis_df["gold_evasion_id"].to_numpy()]
oof_analysis_df["gold_clarity_prob"] = gold_clarity_probs
oof_analysis_df["gold_evasion_prob"] = gold_evasion_probs
oof_analysis_df["gold_clarity_rank"] = (oof_probs_clarity_full > gold_clarity_probs[:, None]).sum(axis=1) + 1
oof_analysis_df["gold_evasion_rank"] = (oof_probs_evasion_full > gold_evasion_probs[:, None]).sum(axis=1) + 1

oof_analysis_df["correct_clarity"] = oof_analysis_df["gold_clarity"] == oof_analysis_df["pred_clarity"]
oof_analysis_df["correct_evasion"] = oof_analysis_df["gold_evasion"] == oof_analysis_df["pred_evasion"]
oof_analysis_df["correct_joint"] = oof_analysis_df["correct_clarity"] & oof_analysis_df["correct_evasion"]

oof_analysis_df["clarity_confusion_cell"] = [
    f"{gold} -> {pred} (gold_id={clarity_label2id[gold]}, pred_id={int(pred_id)})"
    for gold, pred, pred_id in zip(
        oof_analysis_df["gold_clarity"],
        oof_analysis_df["pred_clarity"],
        oof_analysis_df["pred_clarity_id"],
    )
]
oof_analysis_df["evasion_confusion_cell"] = [
    f"{gold} -> {pred} (gold_id={evasion_label2id[gold]}, pred_id={int(pred_id)})"
    for gold, pred, pred_id in zip(
        oof_analysis_df["gold_evasion"],
        oof_analysis_df["pred_evasion"],
        oof_analysis_df["pred_evasion_id"],
    )
]

analysis_dir = Path("../../results/error_analysis")
analysis_dir.mkdir(parents=True, exist_ok=True)

oof_table_path = analysis_dir / "oof_instance_analysis_main_kfold.csv"
legacy_oof_table_path = Path("../../results/oof_instance_analysis_main_kfold.csv")
legacy_oof_table_path.parent.mkdir(parents=True, exist_ok=True)

oof_analysis_df.to_csv(oof_table_path, index=False)
oof_analysis_df.to_csv(legacy_oof_table_path, index=False)

clarity_confusion_counts = (
    oof_analysis_df[oof_analysis_df["gold_clarity"] != oof_analysis_df["pred_clarity"]]
    .groupby(["gold_clarity", "pred_clarity"])
.size()
    .sort_values(ascending=False)
    .reset_index(name="count")
)
evasion_confusion_counts = (
    oof_analysis_df[oof_analysis_df["gold_evasion"] != oof_analysis_df["pred_evasion"]]
    .groupby(["gold_evasion", "pred_evasion"])
.size()
    .sort_values(ascending=False)
    .reset_index(name="count")
)

clarity_classwise_summary = (
    oof_analysis_df.groupby("gold_clarity")
    .agg(
        n=("row_id", "size"),
        accuracy=("correct_clarity", "mean"),
        mean_gold_prob=("gold_clarity_prob", "mean"),
        mean_pred_conf=("pred_clarity_conf", "mean"),
        mean_margin=("pred_clarity_margin", "mean"),
        mean_entropy_norm=("pred_clarity_entropy_norm", "mean"),
    )
    .reset_index()
    .sort_values("n", ascending=False)
)
evasion_classwise_summary = (
    oof_analysis_df.groupby("gold_evasion")
    .agg(
        n=("row_id", "size"),
        accuracy=("correct_evasion", "mean"),
        mean_gold_prob=("gold_evasion_prob", "mean"),
        mean_pred_conf=("pred_evasion_conf", "mean"),
        mean_margin=("pred_evasion_margin", "mean"),
        mean_entropy_norm=("pred_evasion_entropy_norm", "mean"),
    )
    .reset_index()
    .sort_values("n", ascending=False)
)

high_conf_errors = oof_analysis_df[
    (~oof_analysis_df["correct_joint"])
    & (
        (oof_analysis_df["pred_clarity_conf"] >= 0.90)
        | (oof_analysis_df["pred_evasion_conf"] >= 0.90)
    )
].sort_values(["pred_clarity_conf", "pred_evasion_conf"], ascending=False)

low_conf_correct = oof_analysis_df[
    oof_analysis_df["correct_joint"]
    & (
        (oof_analysis_df["pred_clarity_conf"] <= 0.55)
        | (oof_analysis_df["pred_evasion_conf"] <= 0.55)
    )
].sort_values(["pred_clarity_conf", "pred_evasion_conf"], ascending=True)

clarity_confusion_path = analysis_dir / "oof_clarity_confusion_counts.csv"
evasion_confusion_path = analysis_dir / "oof_evasion_confusion_counts.csv"
clarity_classwise_path = analysis_dir / "oof_clarity_classwise_summary.csv"
evasion_classwise_path = analysis_dir / "oof_evasion_classwise_summary.csv"
high_conf_errors_path = analysis_dir / "oof_high_confidence_errors.csv"
low_conf_correct_path = analysis_dir / "oof_low_confidence_correct.csv"

clarity_confusion_counts.to_csv(clarity_confusion_path, index=False)
evasion_confusion_counts.to_csv(evasion_confusion_path, index=False)
clarity_classwise_summary.to_csv(clarity_classwise_path, index=False)
evasion_classwise_summary.to_csv(evasion_classwise_path, index=False)
high_conf_errors.to_csv(high_conf_errors_path, index=False)
low_conf_correct.to_csv(low_conf_correct_path, index=False)

print(f"Saved full OOF analysis table: {oof_table_path}")
print(f"Saved legacy compatibility copy: {legacy_oof_table_path}")
print("Saved additional OOF artifacts:")
for artifact in [
    clarity_confusion_path,
    evasion_confusion_path,
    clarity_classwise_path,
    evasion_classwise_path,
    high_conf_errors_path,
    low_conf_correct_path,
]:
    print(f"  - {artifact}")

print("\nTop evasion confusion cells (off-diagonal):")
print(evasion_confusion_counts.head(12).to_string(index=False))


def shorten_text(text: str, width: int = 88, max_lines: int = 3) -> str:
    if not isinstance(text, str):
        return ""
    compact = " ".join(text.split())
    wrapped = textwrap.wrap(compact, width=width)
    if len(wrapped) <= max_lines:
        return "\n".join(wrapped)
    return "\n".join(wrapped[: max_lines - 1] + [wrapped[max_lines - 1] + " ..."])

def analysis_note(example_type: str, row: pd.Series) -> str:
    if example_type == "Characteristic failure":
        if row["gold_evasion"] == "Partial/half-answer" and row["pred_evasion"] == "Explicit":
            return "The answer begins with direct acknowledgment cues that resemble Explicit behavior, but the response remains only partially committed to the question."
        if row["gold_evasion"] == "Partial/half-answer" and row["pred_evasion"] == "Implicit":
            return "Indirect phrasing and hedging dominate the surface form, so the model leans Implicit while the gold label is Partial/half-answer."
        return "The prediction follows strong surface cues, but misses the deeper pragmatic intent captured by the gold label."
    if example_type == "Correct hard case":
        return "Despite overlapping pragmatic signals, the model resolves both heads correctly in a fine-grained case."
    return "Even with Ambivalent as the dominant clarity class, the model stays on a non-majority clarity decision here."


def pick_first_unique(df: pd.DataFrame, used_ids: set):
    for _, row in df.iterrows():
        rid = int(row["row_id"])
        if rid not in used_ids:
            used_ids.add(rid)
            return row
    return None


hard_evasion_set = {"Partial/half-answer", "Implicit", "Explicit", "Deflection", "Dodging"}
hard_correct_candidates = oof_analysis_df[
    oof_analysis_df["correct_joint"]
    & oof_analysis_df["gold_evasion"].isin(hard_evasion_set)
    & (oof_analysis_df["pred_evasion_conf"] <= 0.80)
].sort_values(["pred_evasion_conf", "pred_clarity_conf"], ascending=[True, True])
if hard_correct_candidates.empty:
    fallback_correct = oof_analysis_df[oof_analysis_df["correct_joint"]].copy()
    fallback_correct["joint_conf"] = (
        fallback_correct["pred_clarity_conf"] + fallback_correct["pred_evasion_conf"]
    ) / 2.0
    hard_correct_candidates = fallback_correct.sort_values("joint_conf", ascending=True)

failure_candidates = oof_analysis_df[
    (oof_analysis_df["gold_evasion"] == "Partial/half-answer")
    & (oof_analysis_df["pred_evasion"].isin(["Explicit", "Implicit"]))
    & (~oof_analysis_df["correct_evasion"])
].sort_values("pred_evasion_conf", ascending=False)
if failure_candidates.empty:
    failure_candidates = oof_analysis_df[~oof_analysis_df["correct_evasion"]].sort_values(
        "pred_evasion_conf", ascending=False
    )

evasion_counts = train_full_df["evasion_label"].value_counts()
rare_evasion_classes = set(evasion_counts[evasion_counts <= evasion_counts.median()].index.tolist())
majority_resist_candidates = oof_analysis_df[
    (oof_analysis_df["gold_clarity"] != "Ambivalent")
    & oof_analysis_df["correct_clarity"]
    & oof_analysis_df["correct_evasion"]
    & oof_analysis_df["gold_evasion"].isin(rare_evasion_classes)
].sort_values(["pred_clarity_conf", "pred_evasion_conf"], ascending=[True, True])
if majority_resist_candidates.empty:
    majority_resist_candidates = oof_analysis_df[
        (oof_analysis_df["gold_clarity"] != "Ambivalent")
        & oof_analysis_df["correct_clarity"]
    ].sort_values("pred_clarity_conf", ascending=True)

used_ids = set()
selected = []

hard_row = pick_first_unique(hard_correct_candidates, used_ids)
if hard_row is not None:
    selected.append(("Correct hard case", hard_row))

failure_row = pick_first_unique(failure_candidates, used_ids)
if failure_row is not None:
    selected.append(("Characteristic failure", failure_row))

resist_row = pick_first_unique(majority_resist_candidates, used_ids)
if resist_row is not None:
    selected.append(("Resists majority-class pull", resist_row))

section53_rows = []
for example_type, row in selected:
    section53_rows.append(
        {
            "example_type": example_type,
            "row_id": int(row["row_id"]),
            "oof_fold": int(row["oof_fold"]),
            "question_excerpt": shorten_text(row["question"], width=88, max_lines=2),
            "answer_excerpt": shorten_text(row["interview_answer"], width=88, max_lines=3),
            "gold_clarity": row["gold_clarity"],
            "pred_clarity": row["pred_clarity"],
            "gold_evasion": row["gold_evasion"],
            "pred_evasion": row["pred_evasion"],
            "pred_clarity_conf": float(row["pred_clarity_conf"]),
            "pred_evasion_conf": float(row["pred_evasion_conf"]),
            "clarity_confusion_cell": row["clarity_confusion_cell"],
            "evasion_confusion_cell": row["evasion_confusion_cell"],
            "analysis_note": analysis_note(example_type, row),
        }
    )

section53_df = pd.DataFrame(section53_rows)
examples_path = analysis_dir / "section_5_3_representative_examples.csv"
legacy_examples_path = Path("../../results/section_5_3_representative_examples.csv")
legacy_examples_path.parent.mkdir(parents=True, exist_ok=True)
section53_df.to_csv(examples_path, index=False)
section53_df.to_csv(legacy_examples_path, index=False)

print("\nRepresentative Section 5.3 examples:")
if section53_df.empty:
    print("No examples selected. Adjust candidate filters in this cell.")
else:
    show_cols = [
        "example_type",
        "row_id",
        "oof_fold",
        "gold_clarity",
        "pred_clarity",
        "gold_evasion",
        "pred_evasion",
        "clarity_confusion_cell",
        "evasion_confusion_cell",
        "question_excerpt",
        "answer_excerpt",
        "analysis_note",
    ]
    print(section53_df[show_cols].to_string(index=False))

print(f"\nSaved representative examples: {examples_path}")
print(f"Saved legacy compatibility copy: {legacy_examples_path}")

In [ ]:
print("Ensemble predictions for QEvasion dev split")
print("="*50)

ensemble_probs_clarity = np.mean(fold_dev_probs_clarity, axis=0)
ensemble_probs_evasion = np.mean(fold_dev_probs_evasion, axis=0)

ensemble_preds_clarity = np.argmax(ensemble_probs_clarity, axis=1)
ensemble_preds_evasion = np.argmax(ensemble_probs_evasion, axis=1)

ensemble_labels_clarity = [clarity_id2label[p] for p in ensemble_preds_clarity]
ensemble_labels_evasion = [evasion_id2label[p] for p in ensemble_preds_evasion]

print(f"Total predictions: {len(ensemble_labels_clarity)}")
print(f"\nClarity predictions distribution:")
print(pd.Series(ensemble_labels_clarity).value_counts())
print(f"\nEvasion predictions distribution:")
print(pd.Series(ensemble_labels_evasion).value_counts())

if "clarity_label" in dev_df.columns:
    valid_clarity_mask = dev_df["clarity_label"].isin(clarity_label2id.keys())
    if valid_clarity_mask.any():
        y_true_dev_clarity = dev_df.loc[valid_clarity_mask, "clarity_label"].tolist()
        y_pred_dev_clarity = np.array(ensemble_labels_clarity)[valid_clarity_mask.values].tolist()
        print("\nDev clarity evaluation")
        print(f"Accuracy: {accuracy_score(y_true_dev_clarity, y_pred_dev_clarity):.4f}")
        print(f"Macro F1: {f1_score(y_true_dev_clarity, y_pred_dev_clarity, average='macro', labels=clarity_labels, zero_division=0):.4f}")
        print("\nClassification Report:")
        print(classification_report(y_true_dev_clarity, y_pred_dev_clarity, labels=clarity_labels, zero_division=0))

annotator_cols = [c for c in ["annotator1", "annotator2", "annotator3"] if c in dev_df.columns]
if annotator_cols:
    pred_evasion_arr = np.array(ensemble_labels_evasion)
    agreement_any = []
    for idx, pred in enumerate(pred_evasion_arr):
        row = dev_df.iloc[idx]
        ann_set = {
            row[c] for c in annotator_cols
            if isinstance(row[c], str) and row[c] in evasion_label2id
        }
        agreement_any.append(pred in ann_set if ann_set else False)
    print(f"\nEvasion any-annotator agreement (dev): {np.mean(agreement_any):.4f}")

    for col in annotator_cols:
        valid_mask = dev_df[col].apply(lambda x: isinstance(x, str) and x in evasion_label2id)
        if valid_mask.any():
            ann_acc = accuracy_score(dev_df.loc[valid_mask, col], pred_evasion_arr[valid_mask.values])
            print(f"Evasion agreement vs {col}: {ann_acc:.4f}")
else:
    print("\nNo annotator columns found in dev split for evasion agreement analysis.")

In [ ]:
import json
import re
from pathlib import Path


def safe_col_suffix(label: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")


analysis_dir = Path("../../results/error_analysis")
analysis_dir.mkdir(parents=True, exist_ok=True)

dev_predictions_df = dev_df.copy()
dev_predictions_df["pred_clarity_id"] = ensemble_preds_clarity
dev_predictions_df["pred_evasion_id"] = ensemble_preds_evasion
dev_predictions_df["pred_clarity"] = ensemble_labels_clarity
dev_predictions_df["pred_evasion"] = ensemble_labels_evasion

dev_predictions_df["pred_clarity_conf"] = ensemble_probs_clarity.max(axis=1)
dev_predictions_df["pred_evasion_conf"] = ensemble_probs_evasion.max(axis=1)

for class_id, class_label in clarity_id2label.items():
    suffix = f"{class_id}_{safe_col_suffix(class_label)}"
    dev_predictions_df[f"prob_clarity_{suffix}"] = ensemble_probs_clarity[:, int(class_id)]

for class_id, class_label in evasion_id2label.items():
    suffix = f"{class_id}_{safe_col_suffix(class_label)}"
    dev_predictions_df[f"prob_evasion_{suffix}"] = ensemble_probs_evasion[:, int(class_id)]

clarity_sorted = np.sort(ensemble_probs_clarity, axis=1)
evasion_sorted = np.sort(ensemble_probs_evasion, axis=1)
dev_predictions_df["pred_clarity_conf_2nd"] = clarity_sorted[:, -2]
dev_predictions_df["pred_evasion_conf_2nd"] = evasion_sorted[:, -2]
dev_predictions_df["pred_clarity_margin"] = dev_predictions_df["pred_clarity_conf"] - dev_predictions_df["pred_clarity_conf_2nd"]
dev_predictions_df["pred_evasion_margin"] = dev_predictions_df["pred_evasion_conf"] - dev_predictions_df["pred_evasion_conf_2nd"]

eps = 1e-12
dev_predictions_df["pred_clarity_entropy"] = -(
    ensemble_probs_clarity * np.log(np.clip(ensemble_probs_clarity, eps, 1.0))
).sum(axis=1)
dev_predictions_df["pred_evasion_entropy"] = -(
    ensemble_probs_evasion * np.log(np.clip(ensemble_probs_evasion, eps, 1.0))
).sum(axis=1)
dev_predictions_df["pred_clarity_entropy_norm"] = dev_predictions_df["pred_clarity_entropy"] / np.log(ensemble_probs_clarity.shape[1])
dev_predictions_df["pred_evasion_entropy_norm"] = dev_predictions_df["pred_evasion_entropy"] / np.log(ensemble_probs_evasion.shape[1])

clarity_top3_idx = np.argsort(ensemble_probs_clarity, axis=1)[:, -3:][:, ::-1]
evasion_top3_idx = np.argsort(ensemble_probs_evasion, axis=1)[:, -3:][:, ::-1]
dev_predictions_df["top3_clarity_labels"] = [
    json.dumps([clarity_id2label[int(i)] for i in idx_row], ensure_ascii=True)
    for idx_row in clarity_top3_idx
]
dev_predictions_df["top3_evasion_labels"] = [
    json.dumps([evasion_id2label[int(i)] for i in idx_row], ensure_ascii=True)
    for idx_row in evasion_top3_idx
]
dev_predictions_df["top3_clarity_probs"] = [
    json.dumps([float(np.round(ensemble_probs_clarity[row_idx, int(i)], 6)) for i in idx_row])
    for row_idx, idx_row in enumerate(clarity_top3_idx)
]
dev_predictions_df["top3_evasion_probs"] = [
    json.dumps([float(np.round(ensemble_probs_evasion[row_idx, int(i)], 6)) for i in idx_row])
    for row_idx, idx_row in enumerate(evasion_top3_idx)
]

annotator_cols = [c for c in ["annotator1", "annotator2", "annotator3"] if c in dev_predictions_df.columns]
if annotator_cols:
    pred_evasion_arr = np.array(ensemble_labels_evasion, dtype=object)
    agree_any = []
    for idx, pred in enumerate(pred_evasion_arr):
        row = dev_predictions_df.iloc[idx]
        ann_set = {
            row[c] for c in annotator_cols
            if isinstance(row[c], str) and row[c] in evasion_label2id
        }
        agree_any.append(pred in ann_set if ann_set else np.nan)
    dev_predictions_df["agree_any_annotator"] = agree_any

    for col in annotator_cols:
        valid_mask = dev_predictions_df[col].apply(lambda x: isinstance(x, str) and x in evasion_label2id)
        dev_predictions_df[f"agree_{col}"] = np.where(
            valid_mask,
            dev_predictions_df[col].to_numpy() == pred_evasion_arr,
            np.nan,
        )

dev_predictions_path = analysis_dir / "qevasion_dev_predictions_multihead_maxpool_kfold_main.csv"
legacy_dev_predictions_path = Path("../../results/qevasion_dev_predictions_multihead_maxpool_kfold_main.csv")
legacy_dev_predictions_path.parent.mkdir(parents=True, exist_ok=True)

dev_predictions_df.to_csv(dev_predictions_path, index=False)
dev_predictions_df.to_csv(legacy_dev_predictions_path, index=False)

summary_payload = {
    "rows": int(len(dev_predictions_df)),
    "clarity_prediction_distribution": pd.Series(ensemble_labels_clarity).value_counts().to_dict(),
    "evasion_prediction_distribution": pd.Series(ensemble_labels_evasion).value_counts().to_dict(),
}

if "clarity_label" in dev_predictions_df.columns:
    valid_clarity_mask = dev_predictions_df["clarity_label"].isin(clarity_label2id.keys())
    if valid_clarity_mask.any():
        y_true = dev_predictions_df.loc[valid_clarity_mask, "clarity_label"].tolist()
        y_pred = np.array(ensemble_labels_clarity)[valid_clarity_mask.values].tolist()
        summary_payload["clarity_accuracy"] = float(accuracy_score(y_true, y_pred))
        summary_payload["clarity_macro_f1"] = float(
            f1_score(y_true, y_pred, average="macro", labels=clarity_labels, zero_division=0)
        )

if annotator_cols:
    summary_payload["annotator_columns"] = annotator_cols
    if "agree_any_annotator" in dev_predictions_df.columns:
        agree_any_series = pd.to_numeric(dev_predictions_df["agree_any_annotator"], errors="coerce")
        summary_payload["evasion_any_annotator_agreement"] = float(agree_any_series.mean())

    for col in annotator_cols:
        agree_col = pd.to_numeric(dev_predictions_df[f"agree_{col}"], errors="coerce")
        summary_payload[f"evasion_agreement_{col}"] = float(agree_col.mean())

dev_summary_path = analysis_dir / "qevasion_dev_predictions_summary_main.json"
with open(dev_summary_path, "w") as f:
    json.dump(summary_payload, f, indent=2)

print(f"Saved rich dev predictions to: {dev_predictions_path}")
print(f"Saved legacy compatibility copy: {legacy_dev_predictions_path}")
print(f"Saved dev summary to: {dev_summary_path}")
print(f"Columns saved: {dev_predictions_df.columns.tolist()}")
print(f"Rows saved: {len(dev_predictions_df)}")

In [ ]:
print(f"Saving best model (from Fold {best_fold})...")

model_save_path = "../../models/multihead_maxpool_kfold_best_dev"
os.makedirs(model_save_path, exist_ok=True)

config = AutoConfig.from_pretrained(MODEL_NAME)
best_model = MultiHeadHierarchicalMaxPooling(config)
best_model.load_state_dict(best_model_state)

torch.save(best_model.state_dict(), os.path.join(model_save_path, "pytorch_model.bin"))

config.save_pretrained(model_save_path)

tokenizer.save_pretrained(model_save_path)

import json
label_mappings = {
    "clarity_label2id": clarity_label2id,
    "clarity_id2label": clarity_id2label,
    "evasion_label2id": evasion_label2id,
    "evasion_id2label": evasion_id2label,
}
with open(os.path.join(model_save_path, "label_mappings.json"), 'w') as f:
    json.dump(label_mappings, f, indent=2)

training_info = {
    "best_fold": best_fold,
    "best_f1_combined": best_f1_combined,
    "n_folds": N_FOLDS,
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "num_clarity_labels": 3,
    "num_evasion_labels": 9,
    "inference_split": "QEvasion test split (used as dev)",
}
with open(os.path.join(model_save_path, "training_info.json"), 'w') as f:
    json.dump(training_info, f, indent=2)

print(f"Model saved to: {model_save_path}")
print(f"Files saved:")
for f in os.listdir(model_save_path):
    print(f"  - {f}")